# Lecture 4: Interventions on Language Models

Can we control a model's behavior through manipulating its internals? This lecture will survey different techniques for ablating a model's activations to identify the information within its components.

### ✍ Learning goals

By the end of the lesson, we hope you will be familiar with different types of interventions we can apply to language models, including

* **Ablations**: structured modifications for removing information in the model.
* **Interchange interventions**: swapping activations between different inputs for identifying causally relevant components.
* **Steering**: interventions optimized to induce behavior in the model.

## 0️⃣ Setup

In [ ]:
from IPython.display import clear_output
import plotly.io as pio

try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    pio.renderers.default = "colab"
else:
    pio.renderers.default = "plotly_mimetype+png"
    !uv sync
    !plotly_get_chrome -y

clear_output()

## 1️⃣ Ablations: identifying information by removing it

### Nullspace projection

1. Gender bias in LLMs
2. Probe for gender
3. Project to probe's nullspace & repeat
4. Test gender bias
5. Plot tSNE before & after

https://github.com/shauli-ravfogel/nullspace_projection/blob/master/notebooks/word_vectors_debiasing.ipynb

### Mean, zero-out, and random ablations

average / zero-out / add noise, check gender bias

do this *only* over biased word, plot success over layers

## 2️⃣ Interchange interventions: what's causally relevant?

In the previous section, we looked at a range of methods that try to remove information from a model's activations to understand where the model stores this information. But does the model use the information we find?

In this section, we'll look at a toy neural network that computes the max of two positive numbers. What we'll see is that removing information doesn't always equate to deleting it from the model's memory. In other words, external interventions that assume a certain structure in the model might have unexpected consequences for the resulting behavior.

### Toy example: neural network for `max`

Let's set up a small neural network that takes in two positive numbers, $x$ and $y$, and returns $\text{max}(x, y)$. Since $\text{max}$ is non-linear, the model must rely on its activation functions to compute it.

Our `MaxNN` uses the $\text{ReLU}$ function like an `if` statement: when $x > y$, $\text{ReLU}(x - y)$ is positive; but when $x < y$, $\text{ReLU}(x - y) = 0$. This means that $0$ has a special value for our model -- it's a flag for which number to select as the maximum.

In [ ]:
# define a neural network that computes the max of two positive numbers
import torch

class MaxNN(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.l1 = torch.nn.Linear(2, 4, bias=False)
    self.l2 = torch.nn.Linear(4, 2, bias=False)
    self.l3 = torch.nn.Linear(2, 1, bias=False)
    self.relu1 = torch.nn.ReLU()
    self.relu2 = torch.nn.ReLU()

    self.l1.weight.data = torch.tensor([
        [1., -1.],  # x - y
        [-1., 1.],  # y - x
        [1., 0.],   # x
        [0., 1.]    # y
    ])

    self.l2.weight.data = torch.tensor([
        [-100., 0., 0., 1.],  # y if x < y else 0
        [0., -100., 1., 0.]   # x if y < x else 0
    ])

    self.l3.weight.data = torch.tensor([
        [1., 1.]
    ])

  def forward(self, x):
    x = self.l1(x)
    x = self.relu1(x)
    x = self.l2(x)
    x = self.relu2(x)
    x = self.l3(x)
    return x

Let's test out our model: play around with different inputs and see if you can detect any edge cases!

In [ ]:
maxnn = MaxNN()

# play around with the input numbers!
inputs = torch.tensor([1., 3.])

maxnn(inputs).item()

3.0

*Note: there's a few edge cases, but for our analysis they aren't important. Going forward, let's assume all inputs are positive, and that there's always a maximum value.*

How does our model know which number is the maximum? A first step might be to apply our lesson from the previous section, and intervene on the model's activations to identify where the maximum value is stored.

Let's focus on zero-out interventions, and where they can go wrong. As we'll see, depending on where we choose to intervene, zeroing out model components can have wildly different consequences on its behavior!

### Zero-out interventions

Let's recall how to use `nnsight` to intervene on a model's activation during a forward run. To start, let's zero out the third layer's activation. Unsurprisingly, doing this will always return zero, because this is the last layer of the model.

In [ ]:
import nnsight

model = nnsight.NNsight(MaxNN())

In [ ]:
# zero out the output of the last layer
inputs = torch.tensor([1., 3.])

original_output = model(inputs)

with model.trace(inputs):
  layer3 = model.l3.output # get layer 3's activation
  layer3[0] = 0. # set the first value in layer 3 to 0
  intervention_output = model.output.save() # save the model's output

print("Before:", original_output.item())
print("After:", intervention_output.item())

Before: 3.0
After: 0.0


Let's get more interesting: can we erase the output information already in the first layer?

In [ ]:
# zero out a neuron in the first layer's output
inputs = torch.tensor([1., 3.])

original_output = model(inputs)

with model.trace(inputs):
  layer1 = model.l1.output # get layer 1's activation
  layer1[-1] = 0. # set the last value in layer 1 to 0
  intervention_output = model.output.save() # save the model's output

print("Before:", original_output.item())
print("After:", intervention_output.item())

Before: 3.0
After: 0.0


### ✏ **Exercise 1**

Looks like zeroing out the last neuron in the first layer changes the output of our model to zero! Is this always the case?

Find 2 more inputs where the intervention has the same effect (that is, the output is 0 after the intervention) and 2 inputs where the intervention has a **different** effect.

*Hint: Play around with different inputs! What pattern do you notice?*

In [ ]:
# copy-pasted code for convenience

######### YOUR CODE HERE ##########
inputs = torch.tensor([1., 3.])
###################################

original_output = model(inputs)

with model.trace(inputs):
  layer1 = model.l1.output # get layer 1's activation
  layer1[-1] = 0. # set the last value in layer 1 to 0
  intervention_output = model.output.save() # save the model's output

print("Before:", original_output.item())
print("After:", intervention_output.item())

Before: 3.0
After: 0.0


What were the inputs where the zero-out intervention had the **same effect**? What were the inputs where the intervention had a **different effect**? What was the new effect, and what did you change about the inputs to achieve it?

> FILL IN YOUR ANSWER HERE

### Intervening on multiple neurons

Looks like, depending on where we intervene on the first layer, zeroing out activations either:

* erases the answer (the model outputs 0)
* doesn't do anything! (the model's output stays the same)

What happens if we intervene on multiple neurons? Can that tell us anything useful?

Let's mysteriously intervene on the first two neurons. What happens?

In [ ]:
# zero out multiple neurons
inputs = torch.tensor([1., 3.])

original_output = model(inputs)

with model.trace(inputs):
  layer1 = model.l1.output # get layer 1's activation
  layer1[:2] = 0. # intervene on both neurons
  intervention_output = model.output.save() # save the model's output

print("Before:", original_output.item())
print("After:", intervention_output.item())

Before: 3.0
After: 4.0


Strange! It seems that the model forgot all about $\max$ and now just adds the inputs together.

What if we intervene on the 2nd and 4th neurons?

In [ ]:
# zero out multiple neurons
inputs = torch.tensor([1., 3.])

original_output = model(inputs)

with model.trace(inputs):
  layer1 = model.l1.output # get layer 1's activation
  layer1[[1, 3]] = 0. # intervene on both neurons
  intervention_output = model.output.save() # save the model's output

print("Before:", original_output.item())
print("After:", intervention_output.item())

Before: 3.0
After: 1.0


Even stranger - the model is now calculating $\min$ instead of $\max$!

We'll figure out *why* this happened later, but for now, let's recap all that can happen when we zero out the model's activations. For our toy example, zero-out interventions can cause the model to:
1. Change its answer to 0.
2. Do nothing.
3. Return the sum of the inputs.
4. Return the smallest input.

Clearly, zeroing out does much more than erasing information in our model. This can often happen in neural networks: thanks to their non-linearity, neural networks can give $0$ a special meaning that's different from just "no information".

To understand our model, we should come up with an intervention that's agnostic to the model's inner workings; we don't want to assume which numbers the model uses to represent certain information, we simply want to know where this information is stored.

That is, instead of stipulating values that the activations might take on, we can **reuse activation values across different inputs**  to test where something is stored in the model! This approach of swapping interventions is usually called **interchange interventions** or **activation patching**.

### Interchange interventions

Let's perform an interchange intervention where we swap the entire value of layer 1. What do you expect the output to be?

In [ ]:
# interchange intervention (from source into base)
source_input = torch.tensor([7., 2.])
base_input = torch.tensor([1., 3.])

# 1. get source activation
with model.trace(source_input):
  source_activation = model.l1.output.save()

# 2. swap base activation with source activation
with model.trace(inputs):
  base_activation = model.l1.output
  base_activation[:] = source_activation[:] # swap values
  intervention_output = model.output.save() # save the model's output


base_output = model(base_input)
source_output = model(source_input)

print("Base output:", base_output.item())
print("Source output:", source_output.item())
print("Intervention output:", intervention_output.item())

Base output: 3.0
Source output: 7.0
Intervention output: 7.0


Perhaps unsurprisingly, the intervention changes the base output to the source output. This is because, after the intervention, the model behaves exactly like it would've behaved if we originally plugged in the source input.

But what if we intervene on only a *part* of the model's activations? This is where things get interesting!

In [ ]:
# interchange intervention on last two neurons

source_input = torch.tensor([7., 2.])
base_input = torch.tensor([1., 3.])

# 1. get source activation
with model.trace(source_input):
  source_activation = model.l1.output.save()

# 2. swap base activation with source activation
with model.trace(inputs):
  base_activation = model.l1.output
  base_activation[2:] = source_activation[2:] # swap values
  intervention_output = model.output.save() # save the model's output


base_output = model(base_input)
source_output = model(source_input)

print("Base output:", base_output.item())
print("Source output:", source_output.item())
print("Intervention output:", intervention_output.item())

Base output: 3.0
Source output: 7.0
Intervention output: 2.0


Interchange intervention on first two neurons.

In [ ]:
# interchange intervention on first two neurons

source_input = torch.tensor([7., 2.])
base_input = torch.tensor([1., 3.])

# 1. get source activation
with model.trace(source_input):
  source_activation = model.l1.output.save()

# 2. swap base activation with source activation
with model.trace(inputs):
  base_activation = model.l1.output
  base_activation[:2] = source_activation[:2] # swap values
  intervention_output = model.output.save() # save the model's output


base_output = model(base_input)
source_output = model(source_input)

print("Base output:", base_output.item())
print("Source output:", source_output.item())
print("Intervention output:", intervention_output.item())

Base output: 3.0
Source output: 7.0
Intervention output: 1.0


We're now starting to get a clearer picture of what's going on! When we intervene on the first two neurons, we change **the function** of the model from $\max$ to $\min$. When we intervene on the last two neurons, we change the **content** of the $\max$ function from the base input to the source inputs.

Putting it together, we can describe how the model computes $\max$:

1. Figure out whether $x > y$ or $y > x$, and store the result in the first two neurons.
2. Store a copy of $x$ and $y$ in the second two neurons.
2. If $x > y$, output the copy of $x$. If $y > x$, output the copy of $y$.

When we intervene, we either intervene on the result of the **comparison**, or on the **intermediate copies** stored by the model.

Interestingly, we figured this out without ever printing out the values of the model's activations! With interchange interventions, it doesn't matter whether the model uses zeroes or other numbers to represent its information. By re-using the model's own activation values, we can understand its computation without needing to make assumptions about its internal representations.

In the next lecture, we'll use interchange interventions to trace how information flows through language models. For the remainder of this lecture, we'll zoom back out and look at how we can directly optimize interventions in order to steer the behavior of language models.

## 3️⃣Steering: controlling models through their internal computations

### Additive interventions

### Representation fine-tuning (ReFT)